In [45]:
endpoint_name='custom-logistic-model-endpoint-48'

# Optional: Test the endpoint
import json
import numpy as np
import boto3

runtime_client = boto3.client('sagemaker-runtime')

# Example data - adjust this based on your model's input requirements
data = {
    "inputs": [[1.0, 2.0, 3.0, 4.0]]  # Replace with your actual input data
}

data = {
    "inputs": [[1.0, 2.0, 3.0, 4.0],[1.0, 1.0, 1.0, 1.0]]  # Replace with your actual input data
}

response = runtime_client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType='application/json',
    Body=json.dumps(data)
)

result = json.loads(response['Body'].read().decode())
print("Prediction result:", result)

Prediction result: {'prediction': [0.320821300824607, 0.2911098274338801]}


In [41]:
import time

start_time=time.time()
num_invocations=1000
test_inputs = [{"inputs": [np.random.random(4).tolist()]} for i in range(num_invocations)]
test_returns = []


for invoke in range(num_invocations):
    response = runtime_client.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='application/json',
        Body=json.dumps(test_inputs[invoke])
    )
    result = json.loads(response['Body'].read().decode())
    test_returns.append(result)
#    result = json.loads(response['Body'].read().decode())

print(f'{num_invocations} invocations in {time.time()-start_time} seconds.')

1000 invocations in 7.122076034545898 seconds.


In [42]:
import random

for i in range(10):
    index = random.choice(np.asarray(range(num_invocations)))
    print(test_inputs[index], '\n', test_returns[index])



{'inputs': [[0.032877751703220603, 0.2902064767938327, 0.7978474063059191, 0.9884324219227837]]} 
 {'prediction': [0.2789161704045072]}
{'inputs': [[0.6178668889008112, 0.9201825781652494, 0.38389720304710806, 0.38181112263239425]]} 
 {'prediction': [0.28074431709042397]}
{'inputs': [[0.25561177294775217, 0.5556322733121192, 0.48916904242652315, 0.09457279911452598]]} 
 {'prediction': [0.2750713886573767]}
{'inputs': [[0.6904103880673589, 0.58689469466889, 0.7713424309285867, 0.4010632938439258]]} 
 {'prediction': [0.28248836771937097]}
{'inputs': [[0.2123435318575173, 0.9206378653395919, 0.539689057001829, 0.5325252486272661]]} 
 {'prediction': [0.2782001820424978]}
{'inputs': [[0.40065093402079344, 0.6762952592788375, 0.2208331791883691, 0.45853347458849214]]} 
 {'prediction': [0.2778740065174536]}
{'inputs': [[0.5235027228758828, 0.46868018068978123, 0.852461114617676, 0.008860722549380196]]} 
 {'prediction': [0.27851834614151966]}
{'inputs': [[0.6445537799317091, 0.5746837728291321

In [ ]:
#concurrency test

In [43]:
import boto3
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

def invoke_endpoint(runtime_client, endpoint_name, input_data):
    try:
        response = runtime_client.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType='application/json',
            Body=json.dumps(input_data)
        )
        return response['Body'].read().decode()
    except Exception as e:
        return str(e)

def burst_endpoint(endpoint_name, num_invocations, concurrency):
    runtime_client = boto3.client('sagemaker-runtime')
    
    # Sample input data - adjust this according to your model's requirements
    input_data = {"inputs": [[1.0, 2.0, 3.0, 4.0]]}
    
    start_time = time.time()
    
    with ThreadPoolExecutor(max_workers=concurrency) as executor:
        futures = [executor.submit(invoke_endpoint, runtime_client, endpoint_name, input_data) 
                   for _ in range(num_invocations)]
        
        for future in as_completed(futures):
            result = future.result()
            # You can add logging here if needed
            # print(result)
    
    end_time = time.time()
    
    print(f"Completed {num_invocations} invocations in {end_time - start_time:.2f} seconds")
    print(f"Average throughput: {num_invocations / (end_time - start_time):.2f} requests/second")

# Usage
num_invocations = 10000
concurrency = 100  # Number of concurrent requests

burst_endpoint(endpoint_name, num_invocations, concurrency)

Completed 10000 invocations in 26.46 seconds
Average throughput: 377.98 requests/second
